# 04 向量化与索引构建（全量）

> **目标**：对全量 `oa_comm_chunks.jsonl`（约 610 万 chunks）构建 BGE + ChromaDB 持久化索引。

> **本 notebook 为全量版**。验证期请用 `vectorize-index.ipynb`（工程内 `data/chroma_db/`）。

## 存储策略

| 模式 | Notebook | 输入 | ChromaDB 位置 |
|------|----------|------|---------------|
| **验证** | `vectorize-index.ipynb` | `data/processed/chunks_sample.jsonl` | 工程内 `data/chroma_db/` |
| **全量（原始）** | 本文件 **【C2】** | `E:\...\oa_comm_chunks.jsonl` | 外接盘 `E:\...\chroma_db\` |
| **全量（工程内加速）** | 本文件 **【C2.5】** | `data/processed/oa_comm_chunks.jsonl` | 工程内 `data/chroma_db/` |

## 推荐执行顺序（工程内加速）

1. **C0 → C1**（环境 + 嵌入模型）
2. **跳过 C2**（保留作外接盘原始方案记录）
3. **【C2.5】** 从 E: 迁移到工程内并续跑建库
4. **C3 → C5** 验证与统计

## 执行前检查

- 外接盘全量 chunks 已就绪（或 C2.5 已复制到工程内）
- CUDA 版 PyTorch + `RESUME=True` 断点续传


---
## 【C0】环境配置 + 设备检测

检测并记录运行设备（GPU/CPU），作为日后对齐的背景信息。

In [1]:
import sys
import os
import json
from pathlib import Path

# Windows：避免 tokenizer 多进程警告；pyarrow 损坏会导致 C1 内核崩溃（见 README）
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

SRC_DIR = Path("../src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

if os.name == "nt":
    DATA_ROOT = Path("E:/med-llm-rag-datasets")
else:
    DATA_ROOT = Path("/Volumes/Lexar/med-llm-rag-datasets")

INPUT_JSONL = DATA_ROOT / "processed" / "oa_comm_chunks.jsonl"
PERSIST_DIR = DATA_ROOT / "chroma_db"
COLLECTION = "pmc_oa_comm_full"

print(f"数据根目录: {DATA_ROOT}")
print(f"输入文件: {INPUT_JSONL}")
print(f"向量库目录: {PERSIST_DIR} (外接盘，全量)")
print(f"collection: {COLLECTION}")
assert DATA_ROOT.exists(), f"数据根目录不存在: {DATA_ROOT}"
assert INPUT_JSONL.exists(), f"全量 chunks 不存在: {INPUT_JSONL}"


数据根目录: E:\med-llm-rag-datasets
输入文件: E:\med-llm-rag-datasets\processed\oa_comm_chunks.jsonl
向量库目录: E:\med-llm-rag-datasets\chroma_db (外接盘，全量)
collection: pmc_oa_comm_full


In [2]:
# 设备检测（背景信息记录）
import torch

ENV_INFO = {
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}
if torch.cuda.is_available():
    ENV_INFO["gpu_name"] = torch.cuda.get_device_name(0)
    ENV_INFO["gpu_mem_GB"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)

print("=== 运行环境 ===")
for k, v in ENV_INFO.items():
    print(f"  {k}: {v}")

if not ENV_INFO["cuda_available"]:
    print("\n⚠️ 当前为 CPU 版 PyTorch，无法用 GPU。")
    print("   全量 610 万 chunks **必须**使用 GPU + CUDA 版 PyTorch，否则耗时极长。")

=== 运行环境 ===
  torch_version: 2.6.0+cu124
  cuda_available: True
  device: cuda
  gpu_name: NVIDIA GeForce RTX 4080 Laptop GPU
  gpu_mem_GB: 12.9


---
## 【C1】加载嵌入模型

加载 `bge-small-en-v1.5`，确认输出维度 = 384。首次运行会联网下载模型权重（约 130MB）。

In [3]:
# 若曾在此内核中 import 过旧版 embedder，请先 Kernel → Restart 再跑 C0
from embedder import DocumentEmbedder
import torch

BATCH_ENCODE = 128 if torch.cuda.is_available() else 64  # 12GB 显存笔记本建议 128
embedder = DocumentEmbedder(model_name="BAAI/bge-small-en-v1.5", batch_size=BATCH_ENCODE)
print(f"encode batch_size: {BATCH_ENCODE}")
for k, v in embedder.device_info().items():
    print(f"  {k}: {v}")
print(f"\n嵌入维度: {embedder.dimension}")
assert embedder.dimension == 384


encode batch_size: 128
  model_name: BAAI/bge-small-en-v1.5
  device: cuda
  torch_version: 2.6.0+cu124
  cuda_available: True
  gpu_name: NVIDIA GeForce RTX 4080 Laptop GPU

嵌入维度: 384


In [4]:
# 快速验证：文档端 vs 查询端编码
doc_vec = embedder.encode_documents(["Diabetes is a chronic metabolic disease."])
qry_vec = embedder.encode_queries(["What is diabetes?"])
print(f"文档向量维度: {len(doc_vec[0])}")
print(f"查询向量维度: {len(qry_vec[0])}")
print(f"查询端已自动加指令前缀: '{embedder.query_instruction}'")

文档向量维度: 384
查询向量维度: 384
查询端已自动加指令前缀: 'Represent this sentence for searching relevant passages: '


---
## 【C2】构建 ChromaDB 索引（外接盘原始方案 · 保留记录）

> **说明**：本节为最初「输入 + 向量库均在 E:」的方案。**若已改在工程内 D: 加速，请跳过 C2，直接运行 【C2.5】**。本节输出（含手动中断报错）保留作对比记录。

分批读取 chunks，文档端编码后写入 ChromaDB（余弦相似度），支持断点续传。

| 参数 | 说明 |
|------|------|
| `BATCH_SIZE` | 每批编码+入库的 chunk 数 |
| `RESUME` | 断点续传（从 progress.json 续跑） |
| `RESET` | 设 True 会**清空**重建该 collection |

In [6]:
BATCH_SIZE = 512
RESUME = True
RESET = False

In [7]:
from index_builder import ChromaIndexBuilder

builder = ChromaIndexBuilder(
    persist_dir=PERSIST_DIR,
    collection_name=COLLECTION,
    embedder=embedder,
)

if RESET:
    builder.client.delete_collection(COLLECTION)
    builder = ChromaIndexBuilder(PERSIST_DIR, COLLECTION, embedder)
    # 同时清除进度文件
    pf = builder._progress_path()
    if pf.exists():
        pf.unlink()
    print("已重置 collection")

print(f"入库前 collection 计数: {builder.collection.count():,}")

入库前 collection 计数: 0


In [8]:
result = builder.build_from_jsonl(
    jsonl_path=INPUT_JSONL,
    batch_size=BATCH_SIZE,
    resume=RESUME,
)
print(f"\n入库完成，collection 共 {result['total_in_collection']:,} 条")

KeyboardInterrupt: 

---
## 【C2.5】工程内路径 + 从 E: 迁移续跑（D: SSD 加速）

> **用途**：外接盘（USB HDD）I/O 成为瓶颈时，将 JSONL + 已有 `chroma_db` 复制到工程目录，在 D: 上续跑。**跳过 C2**，勿重复执行外接盘建库。

| 步骤 | 说明 |
|------|------|
| 1 | 首次：`COPY_FROM_E = True`，从 E: 复制输入与断点向量库 |
| 2 | 复制完成后：改 `COPY_FROM_E = False` |
| 3 | **`REPAIR_HNSW = True`**：若 `count()` 报 `Error loading hnsw index`，自动删损坏 HNSW 段并重建（sqlite 中 15.7 万条 embedding **不丢**） |
| 4 | 全量完成后：整目录拷回 E: 长期存档 |

**前提**：已运行 **C0、C1**。勿与抽样验证共用同一 `chroma_db`（易混 orphan 目录）；全量用 `data/chroma_db_full/`。

In [5]:
# === C2.5a：路径 + 可选从 E: 复制 + HNSW 修复 ===
import shutil
import json
from index_builder import repair_chroma_hnsw

E_ROOT = Path("E:/med-llm-rag-datasets")
E_INPUT = E_ROOT / "processed" / "oa_comm_chunks.jsonl"
E_CHROMA = E_ROOT / "chroma_db"

LOCAL_ROOT = Path("../data").resolve()
LOCAL_INPUT = LOCAL_ROOT / "processed" / "oa_comm_chunks.jsonl"
# 全量专用目录，避免与抽样验证 data/chroma_db/ 混放
LOCAL_CHROMA = LOCAL_ROOT / "chroma_db_full"

COPY_FROM_E = False      # 首次迁移 True；复制完改 False
REPAIR_HNSW = False      # 仅 count 报 hnsw 错时改 True；全量已跑完通常 False（勿误删完好索引）

LOCAL_INPUT.parent.mkdir(parents=True, exist_ok=True)
LOCAL_CHROMA.mkdir(parents=True, exist_ok=True)

if COPY_FROM_E:
    assert E_ROOT.exists(), f"外接盘不存在: {E_ROOT}"
    assert E_INPUT.exists(), f"外接盘 JSONL 不存在: {E_INPUT}"

    if not LOCAL_INPUT.exists():
        print(f"复制 JSONL → {LOCAL_INPUT}（约 9 GB，请耐心等待）…")
        shutil.copy2(E_INPUT, LOCAL_INPUT)
    else:
        print(f"JSONL 已存在，跳过复制: {LOCAL_INPUT}")

    if E_CHROMA.exists():
        print(f"复制 chroma_db → {LOCAL_CHROMA}（含断点 progress）…")
        for item in E_CHROMA.iterdir():
            dest = LOCAL_CHROMA / item.name
            if item.is_dir():
                shutil.copytree(item, dest, dirs_exist_ok=True)
            else:
                shutil.copy2(item, dest)
    else:
        print(f"外接盘尚无 chroma_db，将在工程内新建: {LOCAL_CHROMA}")
else:
    print("COPY_FROM_E=False，跳过复制，直接使用工程内已有数据")

# 若之前误拷到 data/chroma_db/，可改 LOCAL_CHROMA 或手动移入 chroma_db_full/
_legacy = LOCAL_ROOT / "chroma_db"
if not (LOCAL_CHROMA / "chroma.sqlite3").exists() and (_legacy / "chroma.sqlite3").exists():
    print(f"检测到旧路径 {_legacy}，复制到 {LOCAL_CHROMA} …")
    for item in _legacy.iterdir():
        dest = LOCAL_CHROMA / item.name
        if item.is_dir():
            shutil.copytree(item, dest, dirs_exist_ok=True)
        elif not dest.exists():
            shutil.copy2(item, dest)

if REPAIR_HNSW and (LOCAL_CHROMA / "chroma.sqlite3").exists():
    removed = repair_chroma_hnsw(LOCAL_CHROMA, "pmc_oa_comm_full")
    if removed:
        print(f"HNSW 修复：已删除 {len(removed)} 个损坏/ orphan 段目录，embedding 仍在 sqlite 中")
    else:
        print("HNSW 修复：无需处理")

INPUT_JSONL = LOCAL_INPUT
PERSIST_DIR = LOCAL_CHROMA
COLLECTION = "pmc_oa_comm_full"

print("\n=== 工程内全量路径（C2.5 生效）===")
print(f"  INPUT_JSONL: {INPUT_JSONL}")
print(f"  PERSIST_DIR: {PERSIST_DIR}")
print(f"  collection:  {COLLECTION}")
assert INPUT_JSONL.exists(), f"输入不存在: {INPUT_JSONL}"

progress_path = PERSIST_DIR / f"{COLLECTION}.progress.json"
if progress_path.exists():
    with open(progress_path, encoding="utf-8") as f:
        prog = json.load(f)
    print(f"  断点 processed_lines: {prog.get('processed_lines', 0):,}")
else:
    print("  断点: 无（将从头建库）")

# 验证 Chroma 可打开
import chromadb

def _chroma_count():
    client = chromadb.PersistentClient(path=str(PERSIST_DIR))
    return client.get_collection(COLLECTION).count()

try:
    print(f"  collection 计数: {_chroma_count():,}")
except Exception as e:
    if "hnsw" in str(e).lower():
        print("  count 报 hnsw 错，尝试清理 stale index_metadata.pickle …")
        removed = repair_chroma_hnsw(PERSIST_DIR, COLLECTION)
        print(f"  已清理: {len(removed)} 项")
        print(f"  collection 计数（清理后）: {_chroma_count():,}")
    else:
        raise

COPY_FROM_E=False，跳过复制，直接使用工程内已有数据

=== 工程内全量路径（C2.5 生效）===
  INPUT_JSONL: D:\谷歌\04 向量化与索引构建\data\processed\oa_comm_chunks.jsonl
  PERSIST_DIR: D:\谷歌\04 向量化与索引构建\data\chroma_db_full
  collection:  pmc_oa_comm_full
  断点 processed_lines: 6,107,296
  collection 计数: 6,107,296


In [6]:
print("INPUT_JSONL =", INPUT_JSONL)
print("PERSIST_DIR  =", PERSIST_DIR)

INPUT_JSONL = D:\谷歌\04 向量化与索引构建\data\processed\oa_comm_chunks.jsonl
PERSIST_DIR  = D:\谷歌\04 向量化与索引构建\data\chroma_db_full


In [7]:
# === C2.5b：参数 + 建库（等同 C2，但读写工程内路径）===
BATCH_SIZE = 512
RESUME = True
RESET = False  # True 会清空工程内 collection，慎用

from index_builder import ChromaIndexBuilder

builder = ChromaIndexBuilder(
    persist_dir=PERSIST_DIR,
    collection_name=COLLECTION,
    embedder=embedder,
)

if RESET:
    builder.client.delete_collection(COLLECTION)
    builder = ChromaIndexBuilder(PERSIST_DIR, COLLECTION, embedder)
    pf = builder._progress_path()
    if pf.exists():
        pf.unlink()
    print("已重置工程内 collection")

print(f"入库前 collection 计数: {builder.collection.count():,}")

result = builder.build_from_jsonl(
    jsonl_path=INPUT_JSONL,
    batch_size=BATCH_SIZE,
    resume=RESUME,
)
print(f"\n入库完成，collection 共 {result['total_in_collection']:,} 条")
print(f"向量库位置: {PERSIST_DIR.resolve()}")

入库前 collection 计数: 408,064
[续传] 从第 408064 行继续
  已入库 6,107,136 条

入库完成，collection 共 6,107,296 条
向量库位置: D:\谷歌\04 向量化与索引构建\data\chroma_db_full


## builder 初始化 cell
修复c2.5索引表准备跑c3

In [13]:
import importlib, index_builder
importlib.reload(index_builder)
from index_builder import ChromaIndexBuilder, count_embeddings_sqlite

builder = ChromaIndexBuilder(
    persist_dir=PERSIST_DIR,
    collection_name=COLLECTION,
    embedder=embedder,
)
_n = count_embeddings_sqlite(PERSIST_DIR)
print(f"collection 条数（sqlite）: {_n:,}")

collection 条数（sqlite）: 6,107,296


---
## 【C3】保存索引统计

保存到 `outputs/tables/`（全量）。

In [8]:
# 全量库上避免 collection.count() 与 import pandas（Jupyter 下偶发 native 崩溃）
from index_builder import count_embeddings_sqlite

token_counts = []
with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5000:
            break
        token_counts.append(json.loads(line)["token_count"])

token_stats = {
    "mean": round(sum(token_counts) / len(token_counts), 2),
    "max": max(token_counts),
    "min": min(token_counts),
    "note": "基于前 5000 chunks 抽样估计",
}

progress_path = PERSIST_DIR / f"{COLLECTION}.progress.json"
if progress_path.exists():
    with open(progress_path, encoding="utf-8") as pf:
        total_chunks = json.load(pf).get("processed_lines")
else:
    total_chunks = count_embeddings_sqlite(PERSIST_DIR)

print(f"索引条数（progress/sqlite）: {total_chunks:,}")

stats = builder.get_stats(chunk_token_stats=token_stats, total_chunks=total_chunks)
stats["data_type"] = "全量（oa_comm_chunks.jsonl）"
stats["input_file"] = str(INPUT_JSONL)
stats["env_info"] = ENV_INFO
stats["persist_dir"] = str(PERSIST_DIR)

out_path = Path("../outputs/tables/index_stats.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)
print(f"统计已保存: {out_path.resolve()}")


索引条数（progress/sqlite）: 6,107,296
统计已保存: D:\谷歌\04 向量化与索引构建\outputs\tables\index_stats.json


---
## 【C4】相似性检索验证

1. **自相似性**：从索引取一段文本作查询，应命中自身（距离最小）
2. **语义检索**：用自然语言医学问题检索相关片段

In [9]:
with open(INPUT_JSONL, encoding="utf-8") as f:
    sample_rec = json.loads(f.readline())

self_query = sample_rec["text"][:300]
res = builder.query(self_query, n_results=3)
hit = res["ids"][0][0] == sample_rec["chunk_id"]
print(f"自相似性命中自身 ({sample_rec['chunk_id']}): {hit}")


自相似性命中自身 (PMC176545): False


In [10]:
# 语义检索测试：自然语言医学问题
queries = [
    "What is the role of gene regulation in malaria parasites?",
    "conservation of Asian elephants",
    "circadian rhythm in Drosophila",
]
for q in queries:
    res = builder.query(q, n_results=2)
    print(f"\n查询: {q}")
    for cid, dist, title in zip(
        res["ids"][0], res["distances"][0],
        [m.get("source_title", "") for m in res["metadatas"][0]],
    ):
        print(f"  - [{dist:.3f}] {cid}  {title[:70]}")


查询: What is the role of gene regulation in malaria parasites?
  - [0.413] PMC12823243  Gametocidal genes: biological mechanisms owing to hybrid dysgenesis in
  - [0.424] PMC12822987_chunk2  Inapparent maternal ZIKV infection impacts fetal brain development and

查询: conservation of Asian elephants
  - [0.366] PMC12823163  Socio‐Ecological Significance and Anthropogenic Threats to Berlinia (S
  - [0.376] PMC12823164  Divergent Effects of Climate Change on the Potential Habitats of Two M

查询: circadian rhythm in Drosophila
  - [0.414] PMC12823161  An Energetic Tradeoff Best Explains Parturition Timing in Grizzly Bear
  - [0.442] PMC12823063  Anti-resonance in developmental signaling regulates cell fate decision


---
## 【C5】边界情况 + 元数据过滤验证

In [14]:
# 边界：空查询 / 超长查询
validation = {}

try:
    r_empty = builder.query("", n_results=3)
    validation["空查询"] = f"返回 {len(r_empty['ids'][0])} 条（未报错）"
except Exception as e:
    validation["空查询"] = f"异常: {type(e).__name__}"

try:
    r_long = builder.query("diabetes " * 2000, n_results=3)
    validation["超长查询"] = f"返回 {len(r_long['ids'][0])} 条（截断处理，未报错）"
except Exception as e:
    validation["超长查询"] = f"异常: {type(e).__name__}"

for k, v in validation.items():
    print(f"  {k}: {v}")

  空查询: 返回 3 条（未报错）
  超长查询: 返回 3 条（截断处理，未报错）


In [15]:
# 元数据过滤：只在多块文献（strategy=sliding_window）中检索
# 全量 610 万条时 Chroma where= 可能报 Error finding id；index_builder 会自动 over-fetch + Python 过滤
res_filter = builder.query(
    "circadian rhythm",
    n_results=3,
    where_filter={"strategy": "sliding_window"},
)
print("过滤条件: strategy = sliding_window")
print(f"返回 {len(res_filter['ids'][0])} 条:")
for cid, meta in zip(res_filter["ids"][0], res_filter["metadatas"][0]):
    print(f"  - {cid}  strategy={meta.get('strategy')}  total_chunks={meta.get('total_chunks')}")

all_sw = all(m.get("strategy") == "sliding_window" for m in res_filter["metadatas"][0])
print(f"\n元数据过滤生效: {'✅ 是' if all_sw else '⚠️ 否'}")

  [query] Chroma where= 失败，改用 over-fetch + Python 过滤 (InternalError)
过滤条件: strategy = sliding_window
返回 3 条:
  - PMC12823016_chunk1  strategy=sliding_window  total_chunks=3
  - PMC12823016_chunk0  strategy=sliding_window  total_chunks=3
  - PMC12822965_chunk0  strategy=sliding_window  total_chunks=3

元数据过滤生效: ✅ 是


In [16]:
from datetime import datetime

_progress = PERSIST_DIR / f"{COLLECTION}.progress.json"
if _progress.exists():
    with open(_progress, encoding="utf-8") as pf:
        _n_chunks = json.load(pf).get("processed_lines")
else:
    _n_chunks = count_embeddings_sqlite(PERSIST_DIR)

validation_report = {
    "验证日期": datetime.now().isoformat(),
    "数据类型": "全量",
    "collection": COLLECTION,
    "向量数量": _n_chunks,
    "自相似性命中自身": bool(hit),
    "边界情况": validation,
    "元数据过滤生效": bool(all_sw),
}
rep_path = Path("../outputs/tables/query_validation.json")
with open(rep_path, "w", encoding="utf-8") as f:
    json.dump(validation_report, f, indent=2, ensure_ascii=False)
print(f"验证报告: {rep_path.resolve()}")


验证报告: D:\谷歌\04 向量化与索引构建\outputs\tables\query_validation.json


---
## 完成（全量）

| 产物 | 路径 | Git |
|------|------|-----|
| 向量库 | `E:\med-llm-rag-datasets\chroma_db\` | ❌ 外接盘 |
| 索引统计 | `outputs/tables/index_stats.json` | ✅ |
| 查询验证 | `outputs/tables/query_validation.json` | ✅ |
